In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import psycopg2
import json
import pyarrow.parquet as pq

In [ ]:
# ------------------------------
# Add missing columns to reviews table
# ------------------------------
with engine.begin() as conn:
    # Add retrieved_time column (timestamp)
    conn.execute(text("""
        ALTER TABLE reviews 
        ADD COLUMN IF NOT EXISTS retrieved_time TIMESTAMP;
    """))
    
    # Add other_ratings column (integer or text, depending on your data)
    conn.execute(text("""
        ALTER TABLE reviews 
        ADD COLUMN IF NOT EXISTS other_ratings INTEGER;
    """))
    
    print("Added columns to reviews table:")
    print("  - retrieved_time (TIMESTAMP)")
    print("  - other_ratings (INTEGER)")

In [ ]:
# ------------------------------
# Postgres connection
# ------------------------------
user = "ethel"
password = "pa55w0rd13"  # replace with your Postgres password
host = "localhost"
port = "5432"
database = "swimming_pools_db"

engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}")

## the places info

In [ ]:
# ------------------------------
# Path to Parquet
# ------------------------------
parquet_file = "/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/public_swimmingpools_germany/general_poi_json/general_poi_info.parquet"

# ------------------------------
# Columns that are JSONB
# ------------------------------
jsonb_columns = [
    "place_subtype", "opening_hours", "service_options", "higlights",
    "popular_for", "accessibility", "offerings", "dining_options",
    "amenities", "atmosphere", "crowd_level", "planning",
    "payment_options", "children", "parking"
]

# ------------------------------
# Load in chunks
# ------------------------------
chunksize = 5000
pf = pq.ParquetFile(parquet_file)

for batch in pf.iter_batches(batch_size=chunksize):
    chunk = batch.to_pandas()

    # Convert list/dict columns to JSON strings
    for col in jsonb_columns:
        if col in chunk.columns:
            chunk[col] = chunk[col].apply(lambda x: json.dumps(x) if pd.notnull(x) else None)

    # Create WKT geometry from lat/lon
    chunk["geom"] = chunk.apply(
        lambda row: f"SRID=4326;POINT({row['longitude']} {row['latitude']})"
        if pd.notnull(row["latitude"]) and pd.notnull(row["longitude"]) else None,
        axis=1
    )

    # Build upsert query
    insert_columns = list(chunk.columns)
    insert_cols_str = ",".join(insert_columns)
    values_placeholders = ",".join([f":{col}" for col in insert_columns])  # SQLAlchemy named params

    update_cols = [col for col in insert_columns if col != "unique_identifier"]
    update_str = ", ".join([f"{col}=EXCLUDED.{col}" for col in update_cols])

    upsert_query = f"""
        INSERT INTO places ({insert_cols_str})
        VALUES ({values_placeholders})
        ON CONFLICT (unique_identifier) DO UPDATE
        SET {update_str};
    """

    with engine.begin() as conn:
        conn.execute(text(upsert_query), chunk.to_dict(orient="records"))

print("Upsert complete!")

## the reviews info

In [ ]:
# Path to Parquet
#parquet_file = "/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/public_swimmingpools_germany/general_poi_json/few_reviews.parquet"
parquet_file = "/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/public_swimmingpools_germany/clean_datasets/merged_reviews.parquet"
# Columns to convert to JSONB
jsonb_columns = ["other_reviews", "uploaded_pics", "uploaded_pics_links"]

# ------------------------------
# Load in chunks
# ------------------------------
chunksize = 5000
pf = pq.ParquetFile(parquet_file)

batch_count = 0
total_rows = 0

for batch in pf.iter_batches(batch_size=chunksize):
    df = batch.to_pandas()
    
    # Filter out rows where unique_identifier is NULL
    df = df[df['unique_identifier'].notna()]
    
    if len(df) == 0:
        print(f"Batch {batch_count + 1}: All rows skipped (no unique_identifier)")
        batch_count += 1
        continue
    
    # Columns to convert to JSONB
    for col in jsonb_columns:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: json.dumps(x) if pd.notnull(x) else None)
    
    # Set review_time to None since it's relative text like "3 months ago"
    if 'review_time' in df.columns:
        df['review_time'] = None  # Can't parse relative dates
    
    # Build upsert query
    insert_columns = list(df.columns)
    insert_cols_str = ",".join(insert_columns)
    values_placeholders = ",".join([f":{col}" for col in insert_columns])
    update_cols = [col for col in insert_columns if col != "review_id"]
    update_str = ", ".join([f"{col}=EXCLUDED.{col}" for col in update_cols])
    
    upsert_query = f"""
        INSERT INTO reviews ({insert_cols_str})
        VALUES ({values_placeholders})
        ON CONFLICT (review_id) DO UPDATE
        SET {update_str};
    """
    
    # Execute upsert
    with engine.begin() as conn:
        conn.execute(text(upsert_query), df.to_dict(orient="records"))
    
    batch_count += 1
    total_rows += len(df)
    print(f"Processed batch {batch_count} ({len(df)} rows, total: {total_rows})")

print(f"Reviews upsert complete! Total batches: {batch_count}, Total rows: {total_rows}")

## reviews details info

### Create reviews_details table

In [ ]:
# ------------------------------
# Create reviews_details table
# ------------------------------
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS reviews_details (
            place_id TEXT NOT NULL,
            unique_identifier TEXT NOT NULL REFERENCES places(unique_identifier) ON DELETE CASCADE,
            retrieved_time TIMESTAMP,
            json_url TEXT,
            next_key TEXT,
            extracted_reviews INTEGER,
            PRIMARY KEY (place_id, unique_identifier, retrieved_time)
        );
    """))
    
    print("✓ Table 'reviews_details' created successfully!")
    
# Check the table structure
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT column_name, data_type, is_nullable
        FROM information_schema.columns
        WHERE table_name = 'reviews_details'
        ORDER BY ordinal_position;
    """))
    
    print("\nColumns in 'reviews_details' table:")
    print(f"{'Column Name':<30} {'Data Type':<30} {'Nullable':<10}")
    print("-" * 70)
    for row in result:
        print(f"{row[0]:<30} {row[1]:<30} {row[2]:<10}")

In [ ]:
# ------------------------------
# Insert data from merged_details.parquet into reviews_details
# ------------------------------
parquet_file = "/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/public_swimmingpools_germany/clean_datasets/merged_details.parquet"

# Load in chunks
chunksize = 5000
pf = pq.ParquetFile(parquet_file)

batch_count = 0
total_rows = 0

for batch in pf.iter_batches(batch_size=chunksize):
    df = batch.to_pandas()
    
    # Filter out rows where unique_identifier is NULL
    df = df[df['unique_identifier'].notna()]
    
    if len(df) == 0:
        print(f"Batch {batch_count + 1}: All rows skipped (no unique_identifier)")
        batch_count += 1
        continue
    
    # Convert retrieved_time to proper timestamp (if it's a string)
    if 'retrieved_time' in df.columns:
        df['retrieved_time'] = pd.to_datetime(df['retrieved_time'], errors='coerce')
    
    # Build upsert query
    insert_columns = list(df.columns)
    insert_cols_str = ",".join(insert_columns)
    values_placeholders = ",".join([f":{col}" for col in insert_columns])
    
    update_cols = [col for col in insert_columns if col not in ["place_id", "unique_identifier", "retrieved_time"]]
    update_str = ", ".join([f"{col}=EXCLUDED.{col}" for col in update_cols])
    
    upsert_query = f"""
        INSERT INTO reviews_details ({insert_cols_str})
        VALUES ({values_placeholders})
        ON CONFLICT (place_id, unique_identifier, retrieved_time) DO UPDATE
        SET {update_str};
    """
    
    # Execute upsert
    with engine.begin() as conn:
        conn.execute(text(upsert_query), df.to_dict(orient="records"))
    
    batch_count += 1
    total_rows += len(df)
    print(f"Processed batch {batch_count} ({len(df)} rows, total: {total_rows})")

print(f"\n✓ Reviews details upsert complete! Total batches: {batch_count}, Total rows: {total_rows}")